In [ ]:
import threading
import time

SALDO_AWAL = 1_000_000
JUMLAH_THREAD = 10
OPERASI_PER_THREAD = 10_000

# a. Tanpa Lock
saldo = SALDO_AWAL

def tarik_setor():
    # 10.000 kali operasi "tarik 1, lalu setor 1"
    global saldo
    for _ in range(OPERASI_PER_THREAD):
        saldo -= 1
        saldo += 1

print("Tanpa Lock : ")
for percobaan in range(5):
    threads = [threading.Thread(target=tarik_setor) for _ in range(JUMLAH_THREAD)]
    start = time.perf_counter()
    for t in threads: t.start()
    for t in threads: t.join()
    waktu = time.perf_counter() - start
    print(f"Percobaan {percobaan+1}: saldo akhir = {saldo:,} "
          f"({'TETAP' if saldo == SALDO_AWAL else 'BERUBAH!'}), "
          f"waktu = {waktu:.2f} detik")

# b. Dengan Lock
lock = threading.Lock()

def tarik_setor_locked():
    global saldo
    for _ in range(OPERASI_PER_THREAD):
        with lock:              # hanya 1 thread yang boleh mengubah saldo
            saldo -= 1
            saldo += 1

print("\nDengan Lock :")
for percobaan in range(5):
    saldo = SALDO_AWAL
    threads = [threading.Thread(target=tarik_setor_locked) for _ in range(JUMLAH_THREAD)]
    start = time.perf_counter()
    for t in threads: t.start()
    for t in threads: t.join()
    waktu = time.perf_counter() - start
    print(f"Percobaan {percobaan+1}: saldo akhir = {saldo:,} "
          f"({'TETAP' if saldo == SALDO_AWAL else 'BERUBAH!'}), "
          f"waktu = {waktu:.2f} detik")

# Nomor 1. Disini seharusnya nilai saldo akhir yang tanpa lock bsia berbeda dengan SALDO_AWAL nya karena multithreadingnya.
# Tapi disini, saldo akhirnya tidak berubah / tetap karena operasi saldo -= 1 dan saldo += 1 itu berjalan dngan cepat
# yang membuat 1 Thread bisa selesai semua iterasinya sebelum program python pindah ke Thread lain. Juga karena operasi
# iterasinya yg cuma mengurangi 1 dan menambah 1, iterasi itu berjalan dengan sangat cepat dan sekuensial, sampai-sampai
# nilainya bersifat netral. Ini alasan yg membuat peluang corrupt jadi sangat-sangat kecil.

Tanpa Lock : 
Percobaan 1: saldo akhir = 1,000,000 (TETAP), waktu = 0.04 detik
Percobaan 2: saldo akhir = 1,000,000 (TETAP), waktu = 0.04 detik
Percobaan 3: saldo akhir = 1,000,000 (TETAP), waktu = 0.08 detik
Percobaan 4: saldo akhir = 1,000,000 (TETAP), waktu = 0.03 detik
Percobaan 5: saldo akhir = 1,000,000 (TETAP), waktu = 0.02 detik

Dengan Lock :
Percobaan 1: saldo akhir = 1,000,000 (TETAP), waktu = 0.05 detik
Percobaan 2: saldo akhir = 1,000,000 (TETAP), waktu = 0.06 detik
Percobaan 3: saldo akhir = 1,000,000 (TETAP), waktu = 0.13 detik
Percobaan 4: saldo akhir = 1,000,000 (TETAP), waktu = 0.13 detik
Percobaan 5: saldo akhir = 1,000,000 (TETAP), waktu = 0.06 detik


In [ ]:
import threading

# Variabel Global
saldo = 1_000_000
jml_threads = 10
iterasi = 10_000

# 1. Tanpa Lock
def transaksi_tanpa_lock():
    global saldo
    for _ in range(iterasi):
        # Operasi tarik 1 lalu setor 1
        saldo -= 1
        saldo += 1

threads = []

for _ in range(jml_threads):
    t = threading.Thread(target=transaksi_tanpa_lock)
    threads.append(t)
    t.start()

for t in threads:
    t.join()

saldo_akhir_tanpa_lock = saldo
print(f"[Tanpa Lock]  Saldo Akhir: {saldo_akhir_tanpa_lock:,}")

# ==========================================
# 2. SIMULASI DENGAN LOCK (MUTEX)
# ==========================================
lock = threading.Lock()
saldo = 1000000  # Reset saldo awal
def transaksi_dengan_lock():
    global saldo
    for _ in range(iterasi):
        # Mengunci bagian kritis (critical section)
        with lock:
            saldo -= 1
            saldo += 1

saldo = 1000000  # Reset saldo awal
threads = []

for _ in range(jml_threads):
    t = threading.Thread(target=transaksi_dengan_lock)
    threads.append(t)
    t.start()

for t in threads:
    t.join()

saldo_akhir_dengan_lock = saldo
print(f"[Dengan Lock] Saldo Akhir: {saldo_akhir_dengan_lock:,}")